In [1]:
BASE_URL = "s3://stocktwits-nyu" # or local path BASE_URL="local_path"
CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"

In [10]:
import pandas as pd

dtype_map = {"sentiment": "object", "message_id": "object"}
base_url = f"{CSV_URL}/feature_wo_messages"

dfs = {}
max_files = 30  # adjust upward if needed

for i in range(42,43):
    url = f"{base_url}/feature_wo_messages_{i:03d}.csv"
    print("attempting to load", url)
    try:
        df = pd.read_csv(url, dtype=dtype_map)
        name = f"feature_wo_messages_{i:03d}.csv"
        dfs[name] = df
        print("loaded", name)
    except Exception as e:
        print("missing or unreadable:", url, "|", type(e).__name__)
        # continue instead of break in case numbering has gaps
        continue

attempting to load s3://stocktwits-nyu/dataset/v1/data/csv/feature_wo_messages/feature_wo_messages_042.csv


KeyboardInterrupt: 

In [6]:
print("loaded files:", len(dfs))

combined_df = pd.concat(
    [df.assign(source_file=name) for name, df in dfs.items()],
    ignore_index=True
)

loaded files: 7


In [7]:
combined_df

,message_id,user_id,created_at,sentiment,parent_message_id,in_reply_to_message_id,symbol_list,source_file
0,154346073,823564,2019-02-20T14:47:00Z,NaN,154344989.0,154344989.0,[],feature_wo_messages_028.csv
1,154346074,950114,2019-02-20T14:47:01Z,Bullish,NaN,NaN,['JCP'],feature_wo_messages_028.csv
2,154346075,1558352,2019-02-20T14:47:01Z,Bullish,154346075.0,NaN,['NIO'],feature_wo_messages_028.csv
3,154346076,904508,2019-02-20T14:47:02Z,Bullish,NaN,NaN,"['CGC', 'VFF.CA', 'CRON', 'TLRY', 'ACB']",feature_wo_messages_028.csv
4,154346077,1014709,2019-02-20T14:47:02Z,NaN,154346077.0,NaN,['ABIO'],feature_wo_messages_028.csv
...,...,...,...,...,...,...,...,...
14248275,167763886,711796,2019-06-18T01:38:02Z,NaN,167758833.0,167760485.0,[],feature_wo_messages_034.csv
14248276,167763887,56948,2019-06-18T01:38:04Z,NaN,NaN,NaN,['AMPH'],feature_wo_messages_034.csv
14248277,167763888,1362950,2019-06-18T01:38:04Z,NaN,167763626.0,167763626.0,[],feature_wo_messages_034.csv
14248278,167763889,56948,2019-06-18T01:38:04Z,NaN,NaN,NaN,['GE'],feature_wo_messages_034.csv


In [8]:
combined_df["created_at"] = pd.to_datetime(combined_df["created_at"])

In [9]:
combined_df[combined_df["created_at"] >= "2020-01-01"]

,message_id,user_id,created_at,sentiment,parent_message_id,in_reply_to_message_id,symbol_list,source_file


In [ ]:
combined_df.to_csv("combined_stocktwit_data.csv", index=False)

In [ ]:
import yfinance as yf
import pandas as pd
import requests

In [ ]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://en.wikipedia.org/"
}

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
r = requests.get(url, headers=headers)
spx = pd.read_html(r.text)[0]

/tmp/ipykernel_2053/3083819388.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  spx = pd.read_html(r.text)[0]


In [ ]:
spx_tickers = spx["Symbol"].tolist()

In [ ]:
df_snp = df[df["symbol_list"].apply(lambda symbols: any(sym in spx_tickers for sym in symbols))]

In [ ]:
df_snp = df[df["sentiment"].notna()]

In [ ]:
df_snp["created_at"] = pd.to_datetime(df_snp["created_at"])

/tmp/ipykernel_34088/1157205310.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_snp["created_at"] = pd.to_datetime(df_snp["created_at"])


In [ ]:
df_snp[df_snp["created_at"] >= "2019-01-01"]

,message_id,user_id,created_at,sentiment,parent_message_id,in_reply_to_message_id,symbol_list


In [ ]:
df_snp["sentiment"].value_counts()

sentiment
Bullish    487032
Bearish     79222
Name: count, dtype: int64

In [ ]:
df_snp.to_csv("stocktwit_snp_2012_2017.csv", index=False)